[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_16_Deployment_FastAPI_Docker.ipynb)

# 🚀 Lesson 16 — Deployment: FastAPI + Docker + Cloud

**Course:** Becoming a Practical AI/LLM/Agents Engineer  
**Lesson:** 16 of 23  
**Phase:** 3 — Production & Real-World Systems  
**Date:** 2026-05-15

---

## 🎯 What You'll Learn

You've built amazing AI agents — a RAG pipeline, a multi-agent system, an MCP server, and you've packaged it all as an open-source library. But right now your AI agent only runs on *your machine* inside a notebook. The world can't use it.

**This lesson bridges the gap between "it works locally" and "it's live on the internet."**

By the end, you'll know how to:
1. **Wrap your AI agent in a FastAPI web server** — turning it into a callable HTTP API
2. **Containerize it with Docker** — so it runs identically everywhere
3. **Deploy it to the cloud** — Fly.io or Railway (both free tier available)
4. **Think in production terms** — health checks, env vars, graceful errors

---

## 🧠 The Conceptual Map First

```
Your Notebook (Local)
        │
        ▼
   FastAPI App         ← Exposes your agent as HTTP endpoints
        │              ← POST /research → runs AutoResearcher agent
        ▼              ← GET /health → returns {"status": "ok"}
   Docker Image        ← Packages everything: Python, deps, your code
        │              ← One image = reproducible environment anywhere
        ▼
   Cloud Platform      ← Fly.io / Railway / Render / GCP / AWS
        │              ← Runs your Docker image on a real server
        ▼
   Public URL 🌍       ← https://your-agent.fly.dev/research
```

---

## ⚙️ Why FastAPI (not Flask, not Django)?

FastAPI is the dominant choice for AI APIs in 2026. Here's why:

| Feature | FastAPI | Flask | Django |
|---------|---------|-------|--------|
| **Async support** | ✅ Native | ⚠️ Bolted-on | ⚠️ Bolted-on |
| **Auto docs (Swagger)** | ✅ Built-in | ❌ | ❌ |
| **Pydantic validation** | ✅ Native | ❌ | ❌ |
| **Speed** | 🚀 Very fast | Medium | Slower |
| **AI/ML ecosystem** | ✅ Standard | Legacy | Overkill |

**The killer feature for AI:** FastAPI is *async* — when your LLM call is waiting for Anthropic's API (could be 2-10 seconds), the server isn't blocked. It can handle other requests meanwhile. This is critical for production AI services.

---

## 📦 Setup — Install Dependencies

In [ ]:
# Install everything we need for this lesson
!pip install fastapi uvicorn[standard] anthropic pydantic nest-asyncio httpx -q

print("✅ All packages installed!")

In [ ]:
# Load your Anthropic API key from Colab Secrets
# In Colab: click the 🔑 key icon (left sidebar) → Add secret → ANTHROPIC_API_KEY
from google.colab import userdata
import os

os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
print("✅ API key loaded!")

---

## Part 1 — FastAPI Fundamentals

### What is FastAPI, exactly?

FastAPI is a Python web framework for building APIs. You define **endpoints** (URL paths + HTTP methods), and FastAPI handles all the HTTP plumbing: routing requests, parsing JSON bodies, validating inputs, and serializing responses.

The magic ingredient: **Pydantic models** define your request/response shape, and FastAPI auto-generates:
- Input validation (bad request → 422 error automatically)
- Interactive docs at `/docs` (Swagger UI)
- OpenAPI schema at `/openapi.json`

### The anatomy of a FastAPI app

```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()                      # ← The application object

class QueryRequest(BaseModel):        # ← Input shape (Pydantic)
    query: str
    max_tokens: int = 1000

@app.post("/research")               # ← Route decorator
async def research(req: QueryRequest): # ← async = non-blocking LLM calls
    result = await call_llm(req.query)  # ← your AI logic here
    return {"answer": result}          # ← auto-serialized to JSON
```

That's it. FastAPI handles the rest.

---

In [ ]:
# Let's build our first FastAPI app — a simple AI question-answering API
# We'll run it in the background inside Colab using threading

import threading
import time
import nest_asyncio
nest_asyncio.apply()  # Required to run asyncio in Colab

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import anthropic
import uvicorn

# ── Request/Response Models ──────────────────────────────────────────────────
class AskRequest(BaseModel):
    question: str = Field(..., description="The question to ask the AI", min_length=5)
    system_prompt: Optional[str] = Field(
        default="You are a helpful AI assistant. Answer concisely.",
        description="System prompt to customize the AI's behavior"
    )
    max_tokens: int = Field(default=500, ge=50, le=4000)

class AskResponse(BaseModel):
    answer: str
    model: str
    input_tokens: int
    output_tokens: int

# ── Build the FastAPI App ────────────────────────────────────────────────────
app = FastAPI(
    title="AI Question Answering API",
    description="A simple API that wraps an Anthropic LLM",
    version="1.0.0"
)

client = anthropic.Anthropic()

# Health check endpoint — critical for production deployments
@app.get("/health")
async def health_check():
    """Returns OK if the server is running. Used by load balancers and uptime monitors."""
    return {"status": "ok", "service": "ai-qa-api", "version": "1.0.0"}

# Main AI endpoint
@app.post("/ask", response_model=AskResponse)
async def ask_question(req: AskRequest):
    """
    Ask the AI a question. Returns the answer with token usage stats.
    """
    try:
        message = client.messages.create(
            model="claude-haiku-4-5-20251001",  # Fast + cheap for API serving
            max_tokens=req.max_tokens,
            system=req.system_prompt,
            messages=[{"role": "user", "content": req.question}]
        )
        return AskResponse(
            answer=message.content[0].text,
            model=message.model,
            input_tokens=message.usage.input_tokens,
            output_tokens=message.usage.output_tokens
        )
    except anthropic.APIError as e:
        # Never expose raw errors to clients — log internally, return clean message
        raise HTTPException(status_code=502, detail=f"LLM service error: {str(e)}")

print("✅ FastAPI app defined with endpoints: /health and /ask")

In [ ]:
# Run the server in a background thread so Colab doesn't block
# In production you'd just run: uvicorn main:app --host 0.0.0.0 --port 8000

import asyncio

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # Wait for server to start

print("✅ Server running on http://localhost:8000")
print("📖 Swagger docs: http://localhost:8000/docs  (not accessible from Colab, but works locally!)")

In [ ]:
# Test the API with httpx — the async-friendly HTTP client
import httpx
import json

BASE_URL = "http://localhost:8000"

# Test 1: Health Check
print("=" * 50)
print("TEST 1: Health Check")
response = httpx.get(f"{BASE_URL}/health")
print(f"Status: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

# Test 2: Ask a question
print("\n" + "=" * 50)
print("TEST 2: Ask Question")
payload = {
    "question": "What is the difference between a transformer and an RNN in 2 sentences?",
    "max_tokens": 200
}
response = httpx.post(f"{BASE_URL}/ask", json=payload, timeout=30)
print(f"Status: {response.status_code}")
data = response.json()
print(f"Answer: {data['answer']}")
print(f"Tokens used: {data['input_tokens']} in / {data['output_tokens']} out")

# 💡 EXPERIMENT: Try a custom system prompt
# payload = {
#     "question": "What is Docker?",
#     "system_prompt": "You are a pirate who explains technology. Use pirate language.",
#     "max_tokens": 150
# }
# response = httpx.post(f"{BASE_URL}/ask", json=payload, timeout=30)
# print(response.json()["answer"])

In [ ]:
# Test 3: Input Validation — FastAPI auto-validates based on your Pydantic model
print("TEST 3: Validation Error (question too short)")
bad_payload = {"question": "Hi"}  # min_length=5 → should fail
response = httpx.post(f"{BASE_URL}/ask", json=bad_payload)
print(f"Status: {response.status_code}")
print(f"Error: {json.dumps(response.json(), indent=2)}")

# Notice: You wrote ZERO validation code. FastAPI + Pydantic handled everything.

---

## Part 2 — Building an Agent API (Multi-Turn + Tool Use)

The `/ask` endpoint above is stateless (one question, one answer). Real AI agent APIs often need to:
- Support **multi-turn conversations** (pass a conversation ID)
- Execute **tool calls** and return results
- Handle **streaming** (we'll cover that in Lesson 19)

Let's build a more realistic **Research Agent API** that takes a topic and returns a structured research report.

---

In [ ]:
# A more production-realistic agent API
# This wraps a mini ReAct agent that can search (simulated) and summarize

from fastapi import FastAPI, HTTPException, BackgroundTasks
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
from datetime import datetime
import uuid

# ── Models ───────────────────────────────────────────────────────────────────

class ResearchRequest(BaseModel):
    topic: str = Field(..., min_length=3, description="Topic to research")
    depth: str = Field(default="brief", pattern="^(brief|detailed)$")

class ResearchResponse(BaseModel):
    job_id: str
    topic: str
    summary: str
    key_points: List[str]
    created_at: str
    tokens_used: int

class ErrorResponse(BaseModel):
    error: str
    detail: Optional[str] = None

# ── Agent Logic ──────────────────────────────────────────────────────────────

RESEARCH_TOOLS = [
    {
        "name": "search_web",
        "description": "Search the web for information on a topic. Returns relevant snippets.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_wiki_summary",
        "description": "Get a Wikipedia summary for a topic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {"type": "string"}
            },
            "required": ["topic"]
        }
    }
]

def execute_tool(tool_name: str, tool_input: Dict[str, Any]) -> str:
    """Simulated tool execution (in production, this calls real APIs)"""
    if tool_name == "search_web":
        query = tool_input["query"]
        return f"[Search results for '{query}']: Found 5 relevant articles. Key points: transformers use attention mechanisms, published in 2017 by Google, revolutionized NLP, now used in vision (ViT), audio (Whisper), and multimodal models."
    elif tool_name == "get_wiki_summary":
        topic = tool_input["topic"]
        return f"[Wikipedia: {topic}]: {topic} is a machine learning architecture that uses self-attention mechanisms to process sequential data in parallel, unlike RNNs which process sequentially. Introduced in 'Attention Is All You Need' (Vaswani et al., 2017)."
    return "Tool not found"

def run_research_agent(topic: str, depth: str) -> Dict[str, Any]:
    """A mini ReAct agent that researches a topic using tools"""
    client = anthropic.Anthropic()
    total_tokens = 0
    
    system = """You are a research assistant. Use your tools to research the given topic.
    After gathering information with tools, provide:
    1. A clear summary paragraph
    2. Exactly 3 key bullet points
    Format your final answer as:
    SUMMARY: <summary here>
    KEY POINTS:
    - <point 1>
    - <point 2>
    - <point 3>"""
    
    messages = [{"role": "user", "content": f"Research this topic {'in depth' if depth == 'detailed' else 'briefly'}: {topic}"}]
    
    # Agent loop (max 5 iterations)
    for _ in range(5):
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            system=system,
            tools=RESEARCH_TOOLS,
            messages=messages
        )
        total_tokens += response.usage.input_tokens + response.usage.output_tokens
        
        if response.stop_reason == "end_turn":
            # Extract final text
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text = block.text
                    break
            
            # Parse the structured output
            summary = ""
            key_points = []
            
            if "SUMMARY:" in final_text:
                lines = final_text.split("\n")
                in_key_points = False
                for line in lines:
                    if line.startswith("SUMMARY:"):
                        summary = line.replace("SUMMARY:", "").strip()
                    elif "KEY POINTS:" in line:
                        in_key_points = True
                    elif in_key_points and line.strip().startswith("-"):
                        key_points.append(line.strip().lstrip("- "))
            else:
                summary = final_text[:500]
                key_points = ["See summary for details"]
            
            return {"summary": summary, "key_points": key_points, "tokens": total_tokens}
        
        elif response.stop_reason == "tool_use":
            # Execute tools and continue the loop
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
    
    return {"summary": "Research completed.", "key_points": ["No structured output"], "tokens": total_tokens}


# ── FastAPI App v2 ────────────────────────────────────────────────────────────

app2 = FastAPI(title="Research Agent API", version="2.0.0")

@app2.get("/health")
async def health():
    return {"status": "ok"}

@app2.post("/research", response_model=ResearchResponse)
async def research(req: ResearchRequest):
    """Run the research agent on a topic and return structured results."""
    try:
        result = run_research_agent(req.topic, req.depth)
        return ResearchResponse(
            job_id=str(uuid.uuid4())[:8],
            topic=req.topic,
            summary=result["summary"],
            key_points=result["key_points"],
            created_at=datetime.utcnow().isoformat(),
            tokens_used=result["tokens"]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print("✅ Research Agent API defined!")

In [ ]:
# Run Research Agent API on port 8001 (so it doesn't conflict with our first server)

def run_agent_server():
    uvicorn.run(app2, host="0.0.0.0", port=8001, log_level="warning")

agent_thread = threading.Thread(target=run_agent_server, daemon=True)
agent_thread.start()
time.sleep(2)

print("✅ Research Agent API running on http://localhost:8001")

In [ ]:
# Test the Research Agent API
print("Testing Research Agent API...")

payload = {
    "topic": "Transformer architecture in machine learning",
    "depth": "brief"
}

response = httpx.post("http://localhost:8001/research", json=payload, timeout=60)
print(f"Status: {response.status_code}")

if response.status_code == 200:
    data = response.json()
    print(f"\n📋 Job ID: {data['job_id']}")
    print(f"📌 Topic: {data['topic']}")
    print(f"📝 Summary: {data['summary']}")
    print(f"\n🎯 Key Points:")
    for point in data['key_points']:
        print(f"  • {point}")
    print(f"\n💰 Tokens used: {data['tokens_used']}")
    print(f"🕐 Created at: {data['created_at']}")

# 💡 EXPERIMENT: Try different topics
# payload = {"topic": "LangGraph stateful agents", "depth": "detailed"}
# payload = {"topic": "RAG vs Fine-tuning tradeoffs"}

---

## Part 3 — Containerizing with Docker

### What is Docker and why does it matter for AI?

**The problem Docker solves:** "It works on my machine" is the oldest problem in software. You build your AI agent with Python 3.12, anthropic==0.28.0, chromadb==0.4.22... and the cloud server has Python 3.9 and different package versions. It breaks.

**Docker's solution:** Package your entire runtime — Python version, all libraries, your code, environment variables — into a single **image** (like a snapshot of a perfect environment). Anywhere Docker runs, your image runs identically.

### Key Concepts

```
Dockerfile          → Recipe for building your image
    │
    ▼ docker build
Docker Image        → Immutable snapshot (like a template)
    │
    ▼ docker run
Docker Container    → Running instance of the image (like a process)
    │
    ▼ docker push
Docker Registry     → Where images are stored (Docker Hub, GitHub Container Registry)
    │
    ▼
Cloud Platform      → Pulls your image and runs it
```

### The Dockerfile — your build recipe

Every instruction in a Dockerfile creates a **layer** (cached). This is what enables fast rebuilds: if you only change your code but not your dependencies, Docker reuses the cached dependency layer.

---

In [ ]:
# Let's generate a production-ready Dockerfile for our Research Agent API
# We can't run Docker inside Colab, but we can CREATE all the files
# and you'll use them when you have Docker installed locally or in CI/CD

import os

# Create a project directory structure
os.makedirs("research_agent_api", exist_ok=True)
os.makedirs("research_agent_api/app", exist_ok=True)

# ── main.py — the FastAPI application ────────────────────────────────────────
main_py = '''
import os
import uuid
from datetime import datetime
from typing import List, Optional, Dict, Any

import anthropic
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# ── Models ───────────────────────────────────────────────────────────────────
class ResearchRequest(BaseModel):
    topic: str = Field(..., min_length=3)
    depth: str = Field(default="brief", pattern="^(brief|detailed)$")

class ResearchResponse(BaseModel):
    job_id: str
    topic: str
    summary: str
    key_points: List[str]
    created_at: str
    tokens_used: int

# ── App ──────────────────────────────────────────────────────────────────────
app = FastAPI(title="Research Agent API", version="1.0.0")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env automatically

TOOLS = [
    {
        "name": "search_web",
        "description": "Search the web for information",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"]
        }
    }
]

def execute_tool(name: str, inp: Dict[str, Any]) -> str:
    if name == "search_web":
        return f"[Search results for \'{inp[\'query\']}\': Found relevant information about the topic.]"
    return "Unknown tool"

def run_agent(topic: str, depth: str) -> Dict[str, Any]:
    messages = [{"role": "user", "content": f"Research: {topic}. Depth: {depth}."}]
    total_tokens = 0
    
    for _ in range(5):
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            system="Research the topic using tools. Format output as SUMMARY: ... KEY POINTS: - ...",
            tools=TOOLS,
            messages=messages
        )
        total_tokens += response.usage.input_tokens + response.usage.output_tokens
        
        if response.stop_reason == "end_turn":
            text = next((b.text for b in response.content if hasattr(b, "text")), "No response")
            return {"summary": text[:400], "key_points": ["See summary"], "tokens": total_tokens}
        
        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": execute_tool(block.name, block.input)
                    })
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
    
    return {"summary": "Research complete", "key_points": [], "tokens": total_tokens}

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/research", response_model=ResearchResponse)
async def research(req: ResearchRequest):
    try:
        result = run_agent(req.topic, req.depth)
        return ResearchResponse(
            job_id=str(uuid.uuid4())[:8],
            topic=req.topic,
            summary=result["summary"],
            key_points=result["key_points"],
            created_at=datetime.utcnow().isoformat(),
            tokens_used=result["tokens"]
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
'''

with open("research_agent_api/main.py", "w") as f:
    f.write(main_py)

print("✅ main.py created")

In [ ]:
# ── Dockerfile — the star of Part 3 ─────────────────────────────────────────
dockerfile = '''# Stage 1: Use official Python slim image
# "slim" = smaller image (no dev tools), good for production
FROM python:3.12-slim

# Set working directory inside the container
WORKDIR /app

# Copy ONLY requirements first — this layer gets cached
# If requirements don't change, Docker skips this expensive step on rebuild
COPY requirements.txt .

# Install dependencies
# --no-cache-dir = don't cache pip downloads (keeps image smaller)
RUN pip install --no-cache-dir -r requirements.txt

# Copy the rest of the application code
# Separate from requirements so code changes don't invalidate dep cache
COPY main.py .

# Tell Docker which port the app listens on (documentation; doesn't actually open it)
EXPOSE 8000

# The command that runs when the container starts
# --host 0.0.0.0 = accept connections from outside the container (required!)
# --workers 2 = handle 2 requests concurrently (tune based on CPU cores)
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]
'''

with open("research_agent_api/Dockerfile", "w") as f:
    f.write(dockerfile)

# ── requirements.txt ─────────────────────────────────────────────────────────
requirements = '''fastapi==0.115.0
uvicorn[standard]==0.32.0
anthropic==0.40.0
pydantic==2.10.0
'''

with open("research_agent_api/requirements.txt", "w") as f:
    f.write(requirements)

# ── .dockerignore — exclude unnecessary files from the build context ─────────
dockerignore = '''__pycache__/
*.pyc
*.pyo
.env
.git/
.gitignore
*.md
tests/
.pytest_cache/
'''

with open("research_agent_api/.dockerignore", "w") as f:
    f.write(dockerignore)

print("✅ Dockerfile, requirements.txt, and .dockerignore created")
print("\n📂 Project structure:")
for root, dirs, files in os.walk("research_agent_api"):
    # Skip hidden dirs in display
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace("research_agent_api", '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        print(f'{subindent}{file}')

In [ ]:
# Let's display the Dockerfile with explanation inline
print("=" * 60)
print("DOCKERFILE WALKTHROUGH")
print("=" * 60)

explanations = {
    "FROM python:3.12-slim": "Base image: Python 3.12, slim variant (~130MB vs ~1GB)",
    "WORKDIR /app": "All subsequent commands run from /app inside container",
    "COPY requirements.txt .": "Copy requirements BEFORE code (enables layer caching)",
    "RUN pip install": "Install deps as a separate cached layer",
    "COPY main.py .": "Copy code AFTER deps — code changes don't bust dep cache",
    "EXPOSE 8000": "Documents the port (doesn't actually open it)",
    "CMD": "Default command to run when container starts"
}

with open("research_agent_api/Dockerfile") as f:
    for line in f:
        line = line.rstrip()
        if line and not line.startswith('#'):
            key = line.split()[0] if line.split() else ""
            explanation = explanations.get(key, "")
            if explanation:
                print(f"  {line:<50} ← {explanation}")
            else:
                print(f"  {line}")
        else:
            print(f"  {line}")

In [ ]:
# Docker Compose — for running multiple services together
# Useful when your app has a database, cache (Redis), etc.

docker_compose = '''version: "3.9"

services:
  # Your AI API service
  api:
    build: .                          # Build from Dockerfile in current directory
    ports:
      - "8000:8000"                   # host_port:container_port
    environment:
      - ANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}  # Pass from your .env file
    restart: unless-stopped           # Auto-restart on crash
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s                   # Check every 30s
      timeout: 10s
      retries: 3
      start_period: 10s               # Grace period on startup

  # Optional: Add a Redis cache for conversation history
  # redis:
  #   image: redis:7-alpine
  #   ports:
  #     - "6379:6379"
'''

with open("research_agent_api/docker-compose.yml", "w") as f:
    f.write(docker_compose)

# .env example file
env_example = '''# Copy this to .env and fill in your values
# NEVER commit .env to git!
ANTHROPIC_API_KEY=sk-ant-your-key-here
'''

with open("research_agent_api/.env.example", "w") as f:
    f.write(env_example)

print("✅ docker-compose.yml and .env.example created")
print("\n🐳 Local Docker commands (run these in your terminal):")
print("")
print("  # Build the Docker image")
print("  docker build -t research-agent-api .")
print("")
print("  # Run it locally")
print("  docker run -e ANTHROPIC_API_KEY=$ANTHROPIC_API_KEY -p 8000:8000 research-agent-api")
print("")
print("  # Or use Docker Compose (handles env file automatically)")
print("  cp .env.example .env  # fill in your key")
print("  docker-compose up")

---

## Part 4 — Deploying to the Cloud

### Two great free-tier options in 2026:

| Platform | Free tier | Best for |
|----------|-----------|----------|
| **Fly.io** | 3 shared VMs, 3GB storage | Production-grade, scales well |
| **Railway** | $5/month credit | Simplest UX, great for prototypes |
| **Render** | 750 hrs/month | Easy GitHub integration |

### How deployment works (the mental model)

```
Your Machine:
  1. git push (your code goes to GitHub)
  
Cloud Platform:
  2. Detects your push via webhook
  3. Pulls your code
  4. Runs: docker build -t your-app .
  5. Runs: docker run your-app (with your env vars injected)
  6. Points https://your-app.fly.dev → your container
  7. Routes traffic, handles TLS, scales
```

You never SSH into a server. You never install Nginx. The platform handles all of it.

---

In [ ]:
# Generate the fly.toml config for Fly.io deployment
# This is the ONLY Fly-specific file you need

fly_toml = '''# fly.toml — Fly.io deployment configuration
# Docs: https://fly.io/docs/reference/configuration/

app = "research-agent-api"        # Your app name (must be globally unique on fly.io)
primary_region = "sin"            # Singapore — change to "lax" (LA), "iad" (Virginia), etc.

[build]
  # Fly.io automatically detects your Dockerfile and builds it
  # No extra config needed for Dockerfile-based apps

[env]
  # Non-secret env vars go here
  # SECRET env vars (like ANTHROPIC_API_KEY) go in fly secrets — see deployment steps below
  PORT = "8000"

[http_service]
  internal_port = 8000             # Must match the port in your Dockerfile/uvicorn
  force_https = true               # Auto-redirect HTTP → HTTPS
  auto_stop_machines = true        # Scale to zero when no traffic (saves cost!)
  auto_start_machines = true       # Auto-start on new traffic
  min_machines_running = 0         # 0 = scale to zero (free tier)

[[http_service.checks]]
  grace_period = "10s"             # Wait 10s after boot before checking health
  interval = "30s"                 # Check every 30s
  method = "GET"
  path = "/health"                 # Your health endpoint
  timeout = "5s"

[[vm]]
  cpu_kind = "shared"              # Shared CPU (free tier)
  cpus = 1
  memory_mb = 512                  # 512MB RAM — enough for Claude API calls
'''

with open("research_agent_api/fly.toml", "w") as f:
    f.write(fly_toml)

# Railway uses a Procfile (simpler)
procfile = 'web: uvicorn main:app --host 0.0.0.0 --port $PORT\n'

with open("research_agent_api/Procfile", "w") as f:
    f.write(procfile)

# runtime.txt for Railway
with open("research_agent_api/runtime.txt", "w") as f:
    f.write("python-3.12.0\n")

print("✅ fly.toml, Procfile, and runtime.txt created")

In [ ]:
# Print the complete deployment playbook
print("=" * 70)
print("DEPLOYMENT PLAYBOOK — Step by Step")
print("=" * 70)

playbook = """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OPTION A: Deploy to Fly.io (Recommended)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step 1: Install the Fly CLI
  Mac:    brew install flyctl
  Linux:  curl -L https://fly.io/install.sh | sh
  Win:    powershell -c "iwr https://fly.io/install.ps1 -useb | iex"

Step 2: Sign up and login
  flyctl auth signup    # or: flyctl auth login

Step 3: From your project folder, initialize the app
  cd research_agent_api
  flyctl launch         # Detects Dockerfile, prompts for app name & region
                        # Say NO to postgres, NO to redis for now

Step 4: Set your secret API key (NEVER put this in fly.toml)
  flyctl secrets set ANTHROPIC_API_KEY=sk-ant-your-key-here

Step 5: Deploy!
  flyctl deploy
  # Fly builds your Docker image, pushes it, runs it
  # Takes ~2 minutes on first deploy, ~30s after

Step 6: View your live app
  flyctl open
  # Opens https://research-agent-api.fly.dev in your browser
  # Try: https://research-agent-api.fly.dev/health
  # And: https://research-agent-api.fly.dev/docs (Swagger UI!)

Step 7: Monitor
  flyctl logs            # Tail logs in real-time
  flyctl status          # See machine status


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OPTION B: Deploy to Railway (Even simpler)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step 1: Push your project to GitHub
  git init && git add . && git commit -m "Initial commit"
  gh repo create research-agent-api --public --push

Step 2: Go to https://railway.app → New Project → Deploy from GitHub

Step 3: Select your repo → Railway auto-detects Python/Dockerfile

Step 4: Add environment variable:
  ANTHROPIC_API_KEY = sk-ant-your-key-here

Step 5: Click Deploy — Railway builds and deploys automatically

Step 6: Get your URL from the Railway dashboard


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
After Deployment: Test your live API
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  curl https://your-app.fly.dev/health
  
  curl -X POST https://your-app.fly.dev/research \\
    -H 'Content-Type: application/json' \\
    -d '{"topic": "AI agents in 2025", "depth": "brief"}'
"""

print(playbook)

---

## Part 5 — Production Best Practices

Before you ship an AI API to real users, there are a few critical patterns you need to know.

---

In [ ]:
# Best Practice 1: API Key Authentication
# Your /research endpoint is currently open to the world — anyone can spam it and eat your budget

from fastapi import FastAPI, HTTPException, Depends, Security
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
import secrets
import hashlib

# Simple API key auth — generates and validates bearer tokens
security = HTTPBearer()

# In production, these would come from a database. For now, env var.
VALID_API_KEYS = set([
    os.environ.get("SERVICE_API_KEY", "demo-key-abc123"),
])

def verify_api_key(credentials: HTTPAuthorizationCredentials = Security(security)):
    """Dependency that validates the Bearer token on each request."""
    token = credentials.credentials
    if token not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="Invalid or expired API key",
            headers={"WWW-Authenticate": "Bearer"}
        )
    return token

app3 = FastAPI(title="Secured Research API")

@app3.get("/health")
async def health():  # Health check is public — needed by load balancers
    return {"status": "ok"}

@app3.post("/research")
async def secure_research(
    req: dict,
    api_key: str = Depends(verify_api_key)  # ← This dependency runs first
):
    """Protected endpoint — requires valid API key."""
    return {"message": f"Authorized! Processing research for: {req.get('topic')}", "key_prefix": api_key[:10]}

print("✅ API key auth pattern defined")
print("\n🔐 How to call secured endpoints:")
print('  curl -H "Authorization: Bearer demo-key-abc123" \\')
print('       -X POST https://your-app.fly.dev/research \\')
print('       -d \'{\'topic\': \'AI\'}\'"')

In [ ]:
# Best Practice 2: Rate Limiting
# Prevent a single user from overwhelming your API (and your Anthropic bill)

from fastapi import Request
from collections import defaultdict
from datetime import datetime, timedelta

# Simple in-memory rate limiter (use Redis in production for multi-instance)
class SimpleRateLimiter:
    def __init__(self, max_requests: int = 10, window_seconds: int = 60):
        self.max_requests = max_requests
        self.window = window_seconds
        self.requests: dict = defaultdict(list)
    
    def is_allowed(self, client_id: str) -> tuple[bool, dict]:
        now = datetime.utcnow()
        window_start = now - timedelta(seconds=self.window)
        
        # Clean old requests
        self.requests[client_id] = [
            req_time for req_time in self.requests[client_id]
            if req_time > window_start
        ]
        
        count = len(self.requests[client_id])
        
        if count >= self.max_requests:
            return False, {
                "allowed": False,
                "remaining": 0,
                "retry_after": self.window - (now - self.requests[client_id][0]).seconds
            }
        
        self.requests[client_id].append(now)
        return True, {
            "allowed": True,
            "remaining": self.max_requests - count - 1,
            "limit": self.max_requests
        }

# Test the rate limiter
limiter = SimpleRateLimiter(max_requests=3, window_seconds=10)

print("Simulating 5 rapid requests from the same user:")
for i in range(5):
    allowed, info = limiter.is_allowed("user-123")
    status = "✅ ALLOWED" if allowed else "🚫 BLOCKED"
    print(f"  Request {i+1}: {status} | {info}")

# 💡 EXPERIMENT: Try different max_requests and window_seconds values

In [ ]:
# Best Practice 3: Request Logging Middleware
# Every request to a production API should be logged with timing

from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
import time
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

app4 = FastAPI()

# CORS — allow web browsers to call your API from different domains
app4.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],           # In production: ["https://yourdomain.com"]
    allow_methods=["GET", "POST"],
    allow_headers=["*"],
)

# Logging middleware — runs for EVERY request
@app4.middleware("http")
async def log_requests(request: Request, call_next):
    start_time = time.time()
    
    # Process the request
    response = await call_next(request)
    
    # Log after response
    duration_ms = (time.time() - start_time) * 1000
    logger.info(
        f"{request.method} {request.url.path} "
        f"status={response.status_code} "
        f"duration={duration_ms:.0f}ms "
        f"client={request.client.host if request.client else 'unknown'}"
    )
    
    # Add timing header to response (useful for debugging)
    response.headers["X-Process-Time"] = f"{duration_ms:.0f}ms"
    return response

@app4.get("/test")
async def test():
    return {"ok": True}

print("✅ Logging middleware pattern defined")
print("\nIn production logs you'd see:")
print("  2026-05-15 10:23:01 INFO POST /research status=200 duration=3240ms client=1.2.3.4")
print("  2026-05-15 10:23:05 INFO GET /health status=200 duration=2ms client=10.0.0.1")

In [ ]:
# Best Practice 4: Async vs Sync — when it matters for AI

import time
import asyncio
import httpx

print("Understanding async's impact on AI API concurrency...")
print()

# SYNC version — what happens if you use regular (blocking) LLM calls
def sync_llm_call(i: int):
    time.sleep(2)  # Simulate 2s LLM call
    return f"Response {i}"

start = time.time()
# If 3 users hit your API simultaneously, sync processes them sequentially
results = [sync_llm_call(i) for i in range(3)]
sync_time = time.time() - start
print(f"SYNC (sequential): {sync_time:.1f}s for 3 requests")
print(f"  Each user waited: ~{sync_time/3:.1f}s (terrible for user #3!)")

# ASYNC version — async I/O allows concurrent handling
async def async_llm_call(i: int):
    await asyncio.sleep(2)  # Non-blocking sleep (= non-blocking LLM call in real code)
    return f"Response {i}"

async def run_concurrent():
    start = time.time()
    results = await asyncio.gather(*[async_llm_call(i) for i in range(3)])
    return time.time() - start, results

async_time, async_results = asyncio.run(run_concurrent())
print(f"\nASYNC (concurrent): {async_time:.1f}s for 3 requests")
print(f"  Each user waited: ~{async_time:.1f}s (all served simultaneously!)")
print(f"  Speedup: {sync_time/async_time:.1f}x")
print()
print("💡 This is why FastAPI's async is crucial for AI: LLM calls are I/O-bound")
print("   (waiting for Anthropic's servers), not CPU-bound. Async = free concurrency.")

---

## Part 6 — Final Project Structure + GitHub Actions

Let's assemble everything into a final, deployable project.

---

In [ ]:
# Generate a GitHub Actions CI/CD workflow that:
# 1. Runs tests on every push
# 2. Builds Docker image
# 3. Deploys to Fly.io on merge to main

os.makedirs("research_agent_api/.github/workflows", exist_ok=True)

cicd_workflow = '''name: CI/CD Pipeline

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      
      - name: Install dependencies
        run: pip install -r requirements.txt pytest httpx
      
      - name: Run tests
        run: pytest tests/ -v
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}

  deploy:
    needs: test             # Only deploy if tests pass
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'  # Only deploy from main branch
    
    steps:
      - uses: actions/checkout@v4
      
      - name: Install Fly CLI
        uses: superfly/flyctl-actions/setup-flyctl@master
      
      - name: Deploy to Fly.io
        run: flyctl deploy --remote-only
        env:
          FLY_API_TOKEN: ${{ secrets.FLY_API_TOKEN }}  # Set in GitHub repo secrets
'''

with open("research_agent_api/.github/workflows/deploy.yml", "w") as f:
    f.write(cicd_workflow)

# Create a simple test file
os.makedirs("research_agent_api/tests", exist_ok=True)

tests_code = '''import pytest
from fastapi.testclient import TestClient
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))

# We test without real Anthropic calls using a mock
from unittest.mock import patch, MagicMock
from main import app

client = TestClient(app)

def test_health():
    """Health endpoint should always return 200."""
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "ok"

def test_research_validation():
    """Short topic should fail validation."""
    response = client.post("/research", json={"topic": "AI"})
    assert response.status_code == 422  # Pydantic validation error

@patch("main.client.messages.create")
def test_research_success(mock_create):
    """Valid request should return 200 with expected fields."""
    # Mock the Anthropic response
    mock_response = MagicMock()
    mock_response.stop_reason = "end_turn"
    mock_response.usage.input_tokens = 100
    mock_response.usage.output_tokens = 50
    mock_block = MagicMock()
    mock_block.text = "SUMMARY: AI is transforming software.\nKEY POINTS:\n- Point 1\n- Point 2\n- Point 3"
    mock_response.content = [mock_block]
    mock_create.return_value = mock_response
    
    response = client.post("/research", json={"topic": "Artificial Intelligence trends"})
    assert response.status_code == 200
    data = response.json()
    assert "summary" in data
    assert "key_points" in data
    assert "tokens_used" in data
    assert "job_id" in data
'''

with open("research_agent_api/tests/test_api.py", "w") as f:
    f.write(tests_code)

print("✅ GitHub Actions workflow + tests created")

# Final file tree
print("\n📂 Final project structure:")
for root, dirs, files in os.walk("research_agent_api"):
    dirs[:] = sorted([d for d in dirs if not d.startswith('__')])
    level = root.replace("research_agent_api", '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in sorted(files):
        icon = '🐳' if 'Docker' in file else '⚙️' if file.endswith('.toml') or file.endswith('.yml') else '🐍' if file.endswith('.py') else '📄'
        print(f'{subindent}{icon} {file}')

---

## 🧠 Lesson Summary — What You Learned

You now know how to take an AI agent from a notebook to a **live, public URL**.

**FastAPI** — wraps your agent in HTTP endpoints with auto-validation, auto-docs, and async support out of the box. `@app.post("/research")` + a Pydantic model is all you need.

**Docker** — packages your entire runtime into a portable image. The layer caching trick (copy requirements before code) keeps builds fast. `docker run` anywhere = identical behavior.

**Deployment** — Fly.io and Railway take your Docker image and give you a public HTTPS URL. You set secrets via CLI (never in config files). Health checks tell the platform when your app is ready.

**Production patterns** — API key auth protects your budget, rate limiting prevents abuse, logging middleware gives you observability, and async handles concurrent LLM calls efficiently.

---

## 🏗️ Your Mini-Project

Deploy the Research Agent API to the internet:

1. Copy the `research_agent_api/` folder to your computer
2. `cd research_agent_api && git init && git add . && git commit -m "Initial"`
3. Follow the **Fly.io deployment playbook** from Part 4
4. Test your live URL: `curl https://your-app.fly.dev/health`
5. Share your live API URL — you just shipped your first AI service! 🎉

---

## 🔭 What's Next — Lesson 17: Advanced Evals (RAGAS + LLM-Judge Pipelines)

Your agent is now deployed — but how do you know if it's *good*? How do you know if a code change made your RAG better or worse? Lesson 17 covers:
- **RAGAS** — the standard framework for evaluating RAG pipelines (faithfulness, answer relevancy, context recall)
- **LLM-as-judge** — using Claude to automatically score outputs
- **Regression testing for AI** — catching quality degradation before it reaches users
- **Golden datasets** — how to build ground-truth test sets for your specific domain

---

## 📚 Resources

- [FastAPI Docs](https://fastapi.tiangolo.com/) — best web framework docs ever written
- [Docker Getting Started](https://docs.docker.com/get-started/) — official Docker tutorial
- [Fly.io Docs](https://fly.io/docs/) — especially the FastAPI deployment guide
- [Railway Docs](https://docs.railway.app/) — simpler UX, great for experiments
- [Awesome FastAPI](https://github.com/mjhea0/awesome-fastapi) — curated examples and patterns